<a href="https://colab.research.google.com/github/Decoding-Data-Science/bootcamp/blob/main/labs/Day2_RAG_and_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAY 2 | Give DecoBot the Policy: RAG and a Chat App

**Decoding Data Science | AI Accelerator Bootcamp (CPD UK) | OpenAI version**

| | |
|---|---|
| **Time** | 60 minutes (Sat 28 Nov, 11 AM Dubai) |
| **Goal** | Let the AI answer from a real PDF, then put it in a chat app anyone can use |
| **You will build** | A leave-policy chatbot with a clean web page (Gradio) |
| **File needed** | `DDS_Leave_Policy_v2.pdf` (you upload it in Step 2) |
| **Tools** | Google Colab + OpenAI + LlamaIndex + Gradio |

---

### Where we are

```
Day 1   Prompt and clarity   (done)   DecoBot behaves well, but only knows a few pasted facts
Day 2   RAG + chat app       (today)  DecoBot reads the real policy PDF and chats on a web page
Day 3   SQL + agent          (next)   DecoBot checks each person's leave balance
```

### The whole idea in four words

```
 PDF  --CUT-->  chunks  --NUMBER-->  numbers  --SEARCH-->  best chunks  --ANSWER-->  reply
```

**RAG** = Retrieval-Augmented Generation. First **find** the right paragraphs, then let the AI **answer from them**.

### Six words you need today

| Word | Plain meaning |
|---|---|
| **Token** | A small piece of text. About 3/4 of a word. 300 tokens is about 220 words, roughly two short paragraphs. |
| **Chunk** | One piece of the document, cut to a size we choose (for example 300 tokens). |
| **Embedding** | A list of numbers that stands for the *meaning* of a text. Similar meaning, similar numbers. |
| **Index** | The chunks and their numbers, stored together so we can search them. |
| **Score** | How close a chunk is to your question. Higher is closer. |
| **Prompt** | The instructions we give the AI together with the chunks. |

### Today's plan

| Part | What we do | Minutes |
|---|---|---|
| 0 | Setup: install, upload, connect the AI | 6 |
| A | Load the PDF and cut it into chunks | 6 |
| B | Turn chunks into numbers, **save them to a folder**, search | 12 |
| C | Ask a question and get an answer with sources | 4 |
| D | Build the **Gradio chat app** and make it look professional | 18 |
| E | Write a **professional prompt** | 12 |
| | Wrap-up | 2 |

---
## Part 0 | Setup (6 min)

Three small steps. Run each cell with the play button (or Shift + Enter) and wait for the green tick.

### Step 1: Install the tools

*We install the libraries this notebook needs. LlamaIndex does the RAG work, `pypdf` reads the PDF, and Gradio builds the web page.
The last line prints the versions so everyone in class has the same setup. This takes about one minute.*

In [1]:
%pip install -q llama-index-core llama-index-llms-openai llama-index-embeddings-openai pypdf "gradio>=5,<6"

import importlib.metadata as md
for pkg in ["llama-index-core", "llama-index-llms-openai", "llama-index-embeddings-openai", "pypdf", "gradio"]:
    print(f"{pkg:45s}", md.version(pkg))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.
python-fasthtml 0.14.18 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
llama-index-core                              0.14.26
llama-index-llms-openai                       0.9.0
llama-index-embeddings-openai                 0.8.0
pypdf                                         7.2.0
gradio                                        5.50.0


### Step 2: Upload the policy PDF

*A file picker opens. Choose `DDS_Leave_Policy_v2.pdf`. The file goes into the Colab workspace (the folder icon on the left).*

In [21]:
from google.colab import files

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving DDS_Leave_Policy_v2.pdf to DDS_Leave_Policy_v2 (1).pdf
Uploaded: ['DDS_Leave_Policy_v2 (1).pdf']


### Step 3: Connect the AI

*Two things happen here. First, we read your OpenAI API key from Colab Secrets. Second, we choose two models:
a **chat model** (writes answers) and an **embedding model** (turns text into numbers). Then we say hello to check it works.*

> **Before you run this cell:** click the **key icon** on the left of Colab (Secrets). Add a secret named `OPENAI_API_KEY`
> with your key, and switch on **Notebook access**. Never paste a key into a cell.

In [3]:
import os
from google.colab import userdata

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception as error:
    raise RuntimeError(
        "Colab secret OPENAI_API_KEY not found. Click the key icon on the left of Colab, add a secret named "
        "OPENAI_API_KEY, switch on Notebook access, then run this cell again."
    ) from error

from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

MODEL = "gpt-5.4-mini"                    # the chat model: it writes the answers
EMBED_MODEL = "text-embedding-3-small"    # the embedding model: it turns text into numbers

Settings.llm = OpenAI(model=MODEL, reasoning_effort="none")
Settings.embed_model = OpenAIEmbedding(model=EMBED_MODEL, embed_batch_size=50)

print(Settings.llm.complete("Say hello to the DDS team in one short sentence.").text)

Hello, DDS team!


**Checkpoint.** You should see a short hello message above. If you see an error about the key, fix the secret and run the cell again.

---
## Part A | Load the PDF and cut it into chunks (6 min)

The AI cannot read a whole document every time. So we **cut it into chunks** and search for the best ones.

### Step 4: Load the PDF

*We read every page of the PDF and keep the text. Each page also remembers its page number, so the bot can say where an answer came from.*

In [4]:
from pypdf import PdfReader
from llama_index.core import Document

reader = PdfReader("DDS_Leave_Policy_v2.pdf")
documents = [
    Document(text=page.extract_text().replace("\x7f", "-"), metadata={"page": i + 1})
    for i, page in enumerate(reader.pages)
]

words = sum(len(d.text.split()) for d in documents)
print("Pages loaded      :", len(documents))
print("Words in total    :", words)
print("About this many tokens:", int(words * 1.3))
print("\nFirst 400 characters of page 1:\n")
print(documents[0].text[:400])

Pages loaded      : 4
Words in total    : 1768
About this many tokens: 2298

First 400 characters of page 1:

Leave Policy - Decoding Data Science
Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026
Internal - Synthetic demo document for AI training purposes. Not legal advice.
This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are
approved. It replaces Leave Policy v1 (March 2026) in full. Where UAE Federal Decree-Law No. 33 of 202


### Step 5: Cut the text into chunks

*`SentenceSplitter` cuts the text into chunks of about **300 tokens**. It never cuts in the middle of a sentence.
**Overlap** means each chunk repeats the last 50 tokens of the one before it, so an idea at the border is not lost.*

**Your turn (2 minutes):** change `CHUNK_SIZE` to **150**, run the cell, and look at how many chunks you get. Then try **600**.
Small chunks give more pieces. Big chunks give fewer, longer pieces. **Set it back to 300 before you continue.**

In [5]:
from llama_index.core.node_parser import SentenceSplitter

CHUNK_SIZE = 300      # tokens in one chunk. Try 150 and 600, then set it back to 300
CHUNK_OVERLAP = 50    # tokens repeated between neighbouring chunks

splitter = SentenceSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
nodes = splitter.get_nodes_from_documents(documents)      # LlamaIndex calls a chunk a "node"

print("Number of chunks :", len(nodes))
print("Average words    :", sum(len(n.get_content().split()) for n in nodes) // len(nodes))
print("\n--- Chunk 1 ---")
print(nodes[0].get_content()[:500])
print("\n--- Chunk 2 ---")
print(nodes[1].get_content()[:500])

Number of chunks : 12
Average words    : 168

--- Chunk 1 ---
Leave Policy - Decoding Data Science
Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026
Internal - Synthetic demo document for AI training purposes. Not legal advice.
This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are
approved. It replaces Leave Policy v1 (March 2026) in full. Where UAE Federal Decree-Law No. 33 of 2021 (the UAE
Labour Law) provides a greater benefit, the law prevails. Note: this is a synthetic, trai

--- Chunk 2 ---
Maternity leave: 60 calendar days, full pay, 30 calendar
days' notice before expected date, medical documentation required. Paternity leave: 5 working days within 6 months
of birth, full pay, 30 calendar days' notice where foreseeable, birth certificate required. Parental leave (additional): 5
working days per parent within 6 months of birth, full pay, 14 calendar days' notice, birth certificate required.
Bereavement

---
## Part B | Turn chunks into numbers, save them, and search (12 min)

### Step 6: Turn every chunk into numbers (build the index)

*The embedding model reads each chunk and returns a long list of numbers that stands for its meaning. `VectorStoreIndex` does this for all chunks and keeps them together.
This is the only step that calls the embedding model, and it costs a tiny amount.*

In [6]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, show_progress=True)

numbers = Settings.embed_model.get_text_embedding(nodes[0].get_content())
print("\nOne chunk became a list of", len(numbers), "numbers.")
print("The first five:", [round(x, 4) for x in numbers[:5]])

Generating embeddings:   0%|          | 0/12 [00:00<?, ?it/s]


One chunk became a list of 1536 numbers.
The first five: [-0.0316, 0.0493, 0.0682, 0.0153, 0.0005]


**Checkpoint.** A whole paragraph is now a list of numbers. Two chunks with a similar meaning get similar numbers. That is how search will work.

### Step 7: Save the index to a folder and look inside

*Building the index costs money and time, so we **save it to a folder** and reuse it. The folder is called `decobot_index`.
After saving, we open the files to see what is inside. Click the **folder icon** on the left of Colab to browse it as well.*

In [7]:
import os, json

PERSIST_DIR = "decobot_index"
index.storage_context.persist(persist_dir=PERSIST_DIR)

print("Saved to the folder:", PERSIST_DIR, "\n")
for name in sorted(os.listdir(PERSIST_DIR)):
    size_kb = os.path.getsize(os.path.join(PERSIST_DIR, name)) / 1024
    print(f"{name:32s} {size_kb:8.1f} KB")

Saved to the folder: decobot_index 

default__vector_store.json          367.2 KB
docstore.json                        26.5 KB
graph_store.json                      0.0 KB
image__vector_store.json              0.1 KB
index_store.json                      1.2 KB


In [22]:
# Look inside. File 1: the chunk TEXT (docstore.json)
with open(f"{PERSIST_DIR}/docstore.json") as f:
    docstore = json.load(f)
first_chunk = next(iter(docstore["docstore/data"].values()))["__data__"]
print("TEXT of the first saved chunk:\n", first_chunk["text"][:300], "...\n")

# Look inside. File 2: the NUMBERS (default__vector_store.json)
with open(f"{PERSIST_DIR}/default__vector_store.json") as f:
    vector_store = json.load(f)
vectors = vector_store["embedding_dict"]
first_id = next(iter(vectors))
print("NUMBERS saved for", len(vectors), "chunks")
print("First chunk has", len(vectors[first_id]), "numbers. The first five:", [round(x, 4) for x in vectors[first_id][:5]])

TEXT of the first saved chunk:
 Leave Policy - Decoding Data Science
Document ID: DDS-HR-POL-001 | Version 2.0 | Effective date: 1 July 2026
Internal - Synthetic demo document for AI training purposes. Not legal advice.
This policy defines all leave entitlements at DDS, how leave accrues, how to request it, and how requests are
ap ...

NUMBERS saved for 12 chunks
First chunk has 1536 numbers. The first five: [-0.0289, 0.054, 0.0745, 0.0126, 0.0063]


**What you just saw.** The saved folder is just files: one holds the **text** of the chunks, one holds their **numbers**.
There is no hidden magic. A "vector database" is a smarter, bigger version of this folder.

*Optional:* run the next cell if you want a zip of the folder to download and open on your laptop. Colab deletes its files when the session ends, so downloading is how you keep it.

In [9]:
# Optional: download the saved index as a zip file
import shutil
from google.colab import files

shutil.make_archive("decobot_index", "zip", PERSIST_DIR)
files.download("decobot_index.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Step 8: Load the index back from the folder

*This simulates tomorrow morning: a new session, no building, no embedding cost. We read the saved folder and get the same index back in a second.*

In [10]:
from llama_index.core import StorageContext, load_index_from_storage

storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
index = load_index_from_storage(storage_context)

print("Index loaded from disk. Chunks inside:", len(index.docstore.docs))

Index loaded from disk. Chunks inside: 12


### Step 9: Search the index and read the scores

*We ask a question and look at the **three closest chunks**. The **score** shows how close each chunk is to the question: higher means closer.
There is no AI answer yet. This is only the search.*

In [11]:
retriever = index.as_retriever(similarity_top_k=3)     # top_k = how many chunks we bring back

def show_search(question):
    print("QUESTION:", question, "\n")
    for i, hit in enumerate(retriever.retrieve(question), 1):
        text = " ".join(hit.node.get_content().split())
        print(f"[{i}] score {hit.score:.2f} | page {hit.node.metadata.get('page')} | {text[:200]}...\n")

show_search("How many vacation days do I get?")
show_search("How much notice do I need for 12 days of leave?")

QUESTION: How many vacation days do I get? 

[1] score 0.45 | page 1 | Public holidays: per the official UAE calendar, full pay, not deducted from annual leave. 2. Definitions Leave year: 1 January to 31 December. Working day: Monday to Friday, excluding UAE public holid...

[2] score 0.43 | page 1 | Maternity leave: 60 calendar days, full pay, 30 calendar days' notice before expected date, medical documentation required. Paternity leave: 5 working days within 6 months of birth, full pay, 30 calen...

[3] score 0.43 | page 2 | 3.5 Peak periods During flagship delivery windows (major bootcamps, the annual conference season, and product launches), approvals may be restricted. HR publishes restricted weeks at the start of each...

QUESTION: How much notice do I need for 12 days of leave? 

[1] score 0.63 | page 1 | Longer blocks require CEO approval. - At least 5 of the 22 days must be taken as one continuous block of 5 working days each year, to support genuine rest. - Leave may be taken

**Checkpoint.** The first question says *vacation*, but the policy says *annual leave*. The search still found it, because it matches **meaning**, not exact words.
When an answer is wrong, always look at these chunks first. Wrong chunks in, wrong answer out.

---
## Part C | Ask a question and get an answer (4 min)

### Step 10: Ask a question

*A **query engine** does the full job: search the three best chunks, hand them to the chat model, and return an answer.
We wrap it in one small function, `ask()`, that gives back the **answer** and the **sources**. Our chat app will use this same function.*

In [23]:
def make_engine(prompt=None):
    """Build a query engine. Pass a prompt to control how the AI answers (we do this in Part E)."""
    if prompt is None:
        return index.as_query_engine(similarity_top_k=3)
    return index.as_query_engine(similarity_top_k=3, text_qa_template=prompt)

engine = make_engine()          # the engine the chat app will use

def ask(question):
    """Return (answer, sources) for one question."""
    response = engine.query(question)
    answer = str(response).strip()
    lines = []
    for i, hit in enumerate(response.source_nodes, 1):
        text = " ".join(hit.node.get_content().split())
        lines.append(f"**Source {i}** (page {hit.node.metadata.get('page')}, score {(hit.score or 0):.2f}): {text[:220]}...")
    return answer, "\n\n".join(lines)

for question in ["How much notice do I need to book 12 working days of annual leave?",
                 "What is the dress code at DDS?"]:
    answer, sources = ask(question)
    print("QUESTION:", question)
    print("ANSWER  :", answer, "\n")

QUESTION: How much notice do I need to book 12 working days of annual leave?
ANSWER  : 30 calendar days' notice. 

QUESTION: What is the dress code at DDS?
ANSWER  : The leave policy does not mention any dress code at DDS. 



**Checkpoint.** The first answer should be correct. Now read the answer to the **dress code** question. The policy says nothing about it.
Did the bot say so, or did it guess? We will fix this in Part E with a better prompt.

---
## Part D | Build the Gradio chat app (18 min)

**Gradio** turns a Python function into a web page. You write the function, Gradio gives you the buttons, the text boxes and a link you can share.

### Step 11: Your first Gradio app (5 min)

*`gr.Interface` is the quickest way: give it a function, an input box and an output box. `share=True` makes a public link that also works on your phone for 72 hours.
Run the cell, type a question, and press **Submit**.*

In [13]:
import gradio as gr

def answer_only(question):
    answer, sources = ask(question)
    return answer

try:
    demo.close()                    # closes an older copy if you run this cell twice
except Exception:
    pass

demo = gr.Interface(
    fn=answer_only,
    inputs=gr.Textbox(label="Your leave question", placeholder="For example: How many carry-over days can I have?"),
    outputs=gr.Textbox(label="DecoBot says"),
    title="DecoBot | DDS Leave Assistant",
    description="Ask a question about the DDS leave policy.",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/11/28 09:53:32 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>

**Checkpoint.** You have a working web app with 9 lines of code. It works, but it looks plain, there is no chat, and it does not show where the answer came from.
Let us fix that.

### Step 12: A professional chat app (10 min)

*`gr.Blocks` lets us design the page ourselves. This version adds a header, a real chat window, example questions you can click, a **Sources** panel that shows where each answer came from, and a footer note.
Read the code from top to bottom: each part of the page is one block.*

In [14]:
import gradio as gr

CSS = """
.gradio-container {max-width: 860px !important; margin: auto !important;}
#header {text-align: center; padding: 18px 0 6px 0;}
#header h1 {margin-bottom: 4px;}
#footer {text-align: center; color: #6b7280; font-size: 13px; padding-top: 8px;}
"""

EXAMPLES = [
    "How many annual leave days do I get?",
    "How much notice do I need for 12 working days of leave?",
    "How many days can I carry over to next year?",
    "When do I need a medical certificate?",
]

def respond(message, history):
    answer, sources = ask(message)
    history = history + [{"role": "user", "content": message},
                         {"role": "assistant", "content": answer}]
    return "", history, sources

try:
    app.close()                     # closes an older copy if you run this cell twice
except Exception:
    pass

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", neutral_hue="slate"), css=CSS, title="DecoBot") as app:
    gr.Markdown("# DecoBot\n### Your DDS leave assistant. Ask about the leave policy.", elem_id="header")
    chatbot = gr.Chatbot(type="messages", height=380, label="Chat", show_copy_button=True)
    with gr.Row():
        box = gr.Textbox(placeholder="Type your leave question and press Enter", show_label=False, scale=5)
        send = gr.Button("Ask", variant="primary", scale=1)
    gr.Examples(examples=EXAMPLES, inputs=box, label="Try one of these")
    with gr.Accordion("Where did this answer come from?", open=False):
        sources_box = gr.Markdown("Ask a question to see the policy chunks used.")
    gr.Markdown("Answers come from the DDS Leave Policy v2. For final decisions, contact People & Culture.", elem_id="footer")

    send.click(respond, [box, chatbot], [box, chatbot, sources_box])
    box.submit(respond, [box, chatbot], [box, chatbot, sources_box])

app.launch(share=True)

/tmp/ipykernel_1310/80764642.py:28: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", neutral_hue="slate"), css=CSS, title="DecoBot") as app:
/tmp/ipykernel_1310/80764642.py:28: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", neutral_hue="slate"), css=CSS, title="DecoBot") as app:
/tmp/ipykernel_1310/80764642.py:30: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(type="messages", height=380, label="Chat", show_copy_button=True)
/tmp/ipykernel_1310/80764642.py:30: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from Fal

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/11/28 09:53:33 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-11-28T09:53:33Z is after 2026-11-28T09:01:34Z


<IPython.core.display.Javascript object>

**Try it.** Click an example, then open **Where did this answer come from?**. Ask the dress code question again.
Everything you see (colours, header, footer, examples) is a few lines you can change.

### Make your app more beautiful: starter prompts

You do not need to remember Gradio code. Paste your working code into an AI assistant (Claude, ChatGPT or Gemini) together with **one** of these prompts.
Always add the last two rules, they keep the AI from breaking your app.

**1. Brand look**
```
Here is my working Gradio code. Restyle it for DecoBot, an HR assistant at a company called Decoding Data Science (DDS).
Use gr.themes.Soft with a deep blue primary colour, rounded corners, the Inter font, a centred header with a short subtitle,
and a maximum page width of 860px. use 5.55 edition for gradio
```

**2. Better chat experience**
```
Improve the chat experience in my Gradio app: add a friendly welcome message from DecoBot, a placeholder that tells the user
what to ask, a Clear chat button, a copy button on answers, and 5 clickable example questions about leave.
```

**3. Trust and professionalism**
```
Make my Gradio app look trustworthy for employees: add a short disclaimer in the footer, show the sources in a collapsible panel,
and show a friendly message instead of an error if the AI call fails.
```

**4. Layout with tabs**
```
Redesign my Gradio app with two tabs: "Ask DecoBot" (the chat) and "About" (what DecoBot can and cannot answer, which policy
version it uses, and who to contact in People and Culture).
```

**5. Mobile friendly**
```
Make my Gradio app work well on a phone: single column, large tap targets, readable font size, and no horizontal scrolling.
```

**6. Act as a designer**
```
Act as a UX designer. Review my Gradio app, list the 5 biggest improvements for an HR chatbot used by employees,
then apply them and return the full updated code.
```

**Always add these two lines at the end of any prompt:**
```
Do not change the ask() function or the logic. Use Gradio 5. Return the complete code in one cell that I can paste into Colab.
```

**Your turn (5 minutes).** Copy the code from Step 12, pick one prompt, paste both into an AI assistant, and run the new code in a new cell.
Show your neighbour the before and after.

In [20]:
import gradio as gr

# Custom CSS to force elegant styling, max-width of 860px, Inter font, and responsive layout
CSS = """
.gradio-container {
    max-width: 860px !important;
    margin: auto !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
#header {
    text-align: center;
    padding: 24px 0 12px 0;
}
#header h1 {
    margin-bottom: 6px;
    color: #1e3a8a;
    font-weight: 700;
}
#header p {
    color: #4b5563;
    font-size: 15px;
}
#footer {
    text-align: center;
    color: #6b7280;
    font-size: 12px;
    padding-top: 16px;
    border-top: 1px solid #e5e7eb;
    margin-top: 20px;
}
/* Mobile friendliness adjustments */
@media (max-width: 640px) {
    .gradio-container {
        padding: 10px !important;
    }
    button, input, textarea {
        min-height: 48px !important; /* Larger tap targets */
    }
}
"""

EXAMPLES = [
    "How many annual leave days do I get?",
    "How much notice do I need for 12 working days of leave?",
    "How many days can I carry over to next year?",
    "When do I need a medical certificate?",
    "My grandmother passed away. How many days of leave do I get?"
]

# Initial greeting conversation
WELCOME_HISTORY = [
    {
        "role": "assistant",
        "content": "👋 Hello! I am **DecoBot**, your friendly HR leave assistant at Decoding Data Science (DDS). Ask me any question about our Leave Policy (v2.0), and I'll find the answers for you!"
    }
]

def respond(message, history):
    try:
        # Defensive check and calling the preset ask() function
        answer, sources = ask(message)
    except Exception as e:
        # Show a friendly message instead of a crash trace
        answer = "I'm sorry, I'm having trouble connecting to my brain right now. Please try asking again in a moment or contact People & Culture directly!"
        sources = "Error contacting the AI system."

    # Append new chat round
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return "", history, sources

def clear_session():
    return WELCOME_HISTORY, "Ask a question to see the policy chunks used."

try:
    app.close() # Close any previous running instances safely
except Exception:
    pass

# Configure Soft Theme correctly by directly modifying theme attributes
deco_theme = gr.themes.Soft(
    primary_hue="blue",
    neutral_hue="slate",
    font=["Inter", "sans-serif"]
)

# Directly modifying variables to set rounded corners properly
deco_theme.radius_md = "8px"
deco_theme.radius_lg = "12px"

with gr.Blocks(theme=deco_theme, css=CSS, title="DecoBot | DDS HR Assistant") as app:
    # Centered header with subtitle
    gr.HTML(
        "<div id='header'>"
        "<h1>DecoBot</h1>"
        "<p>Your Decoding Data Science (DDS) HR Leave Assistant</p>"
        "</div>"
    )

    # Setup tabs for user layout
    with gr.Tabs():
        with gr.TabItem("Ask DecoBot"):
            # Chat component with copy button enabled
            chatbot = gr.Chatbot(
                value=WELCOME_HISTORY,
                type="messages",
                height=400,
                label="DecoBot Chat",
                show_copy_button=True
            )

            with gr.Row():
                box = gr.Textbox(
                    placeholder="Ask about carry-over days, maternity notice, sick leave...",
                    show_label=False,
                    scale=4
                )
                send = gr.Button("Ask", variant="primary", scale=1)
                clear = gr.Button("Clear Chat", variant="stop", scale=1)

            # Clickable examples
            gr.Examples(examples=EXAMPLES, inputs=box, label="Frequently Asked Leave Questions")

            # Collapsible sources panel
            with gr.Accordion("🔍 Where did this answer come from? (Sources)", open=False):
                sources_box = gr.Markdown("Ask a question to see the policy chunks used.")

        with gr.TabItem("About"):
            gr.Markdown("""
            ### About DecoBot & DDS Leave Policy

            **What DecoBot Can Answer:**
            * Annual leave entitlement, carry-over rules, and notice periods.
            * Parental leave, maternity, paternity, and bereavement policies.
            * Public holiday policies and medical certificate guidelines.

            **What DecoBot Cannot Answer:**
            * Personal leave balances (Coming tomorrow in Day 3!).
            * Discretionary exceptions outside of the general written policy.

            ---

            **Policy Specifications:**
            * **Document ID:** DDS-HR-POL-001
            * **Version:** 2.0
            * **Effective Date:** 1 July 2026

            ---

            **Need more help?**
            If your question is sensitive or requires manual approval, please reach out directly to the **People and Culture** team via email or Slack.
            """)

    # Trustworthy disclaimer in footer
    gr.Markdown(
        "*Disclaimer: DecoBot provides guidance based strictly on the DDS Leave Policy v2.0. "
        "For official or final determinations, please consult directly with the People & Culture department.*",
        elem_id="footer"
    )

    # Set up events
    send.click(respond, [box, chatbot], [box, chatbot, sources_box])
    box.submit(respond, [box, chatbot], [box, chatbot, sources_box])
    clear.click(clear_session, outputs=[chatbot, sources_box])

app.launch(share=True)


/tmp/ipykernel_1310/3280159101.py:93: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=deco_theme, css=CSS, title="DecoBot | DDS HR Assistant") as app:
/tmp/ipykernel_1310/3280159101.py:93: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=deco_theme, css=CSS, title="DecoBot | DDS HR Assistant") as app:
/tmp/ipykernel_1310/3280159101.py:106: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1310/3280159101.py:106: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your 

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/11/28 10:04:52 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-11-28T10:04:52Z is after 2026-11-28T09:01:34Z


<IPython.core.display.Javascript object>

In [24]:
import gradio as gr

# Custom CSS to force elegant styling, max-width of 860px, Inter font, and responsive layout
CSS = """
.gradio-container {
    max-width: 860px !important;
    margin: auto !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
}
#header {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 16px;
    padding: 24px 0 12px 0;
}
#header img {
    height: 60px;
    width: auto;
}
#header-text {
    text-align: left;
}
#header-text h1 {
    margin: 0;
    color: #1e3a8a;
    font-weight: 700;
    font-size: 28px;
}
#header-text p {
    margin: 4px 0 0 0;
    color: #4b5563;
    font-size: 15px;
}
#footer {
    text-align: center;
    color: #6b7280;
    font-size: 12px;
    padding-top: 16px;
    border-top: 1px solid #e5e7eb;
    margin-top: 20px;
}
/* Mobile friendliness adjustments */
@media (max-width: 640px) {
    .gradio-container {
        padding: 10px !important;
    }
    button, input, textarea {
        min-height: 48px !important; /* Larger tap targets */
    }
    #header {
        flex-direction: column;
        text-align: center;
        gap: 8px;
    }
    #header-text {
        text-align: center;
    }
}
"""

EXAMPLES = [
    "How many annual leave days do I get?",
    "How much notice do I need for 12 working days of leave?",
    "How many days can I carry over to next year?",
    "When do I need a medical certificate?",
    "My grandmother passed away. How many days of leave do I get?"
]

# Initial greeting conversation with HALA BOT branding
WELCOME_HISTORY = [
    {
        "role": "assistant",
        "content": "👋 Hello! I am **HALA BOT**, your friendly HR leave assistant at Decoding Data Science (DDS). Ask me any question about our Leave Policy (v2.0), and I'll find the answers for you!"
    }
]

def respond(message, history):
    try:
        # Defensive check and calling the preset ask() function
        answer, sources = ask(message)
    except Exception as e:
        # Show a friendly message instead of a crash trace
        answer = "I'm sorry, I'm having trouble connecting to my brain right now. Please try asking again in a moment or contact People & Culture directly!"
        sources = "Error contacting the AI system."

    # Append new chat round
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer}
    ]
    return "", history, sources

def clear_session():
    return WELCOME_HISTORY, "Ask a question to see the policy chunks used."

try:
    app.close() # Close any previous running instances safely
except Exception:
    pass

# Configure Soft Theme correctly by directly modifying theme attributes
deco_theme = gr.themes.Soft(
    primary_hue="blue",
    neutral_hue="slate",
    font=["Inter", "sans-serif"]
)

# Directly modifying variables to set rounded corners properly
deco_theme.radius_md = "8px"
deco_theme.radius_lg = "12px"

with gr.Blocks(theme=deco_theme, css=CSS, title="HALA BOT | DDS HR Assistant") as app:
    # Centered header containing both Logo and Title side-by-side
    gr.HTML(
        "<div id='header'>"
        "<img src='https://github.com/Decoding-Data-Science/airesidency/blob/main/dds-logo-removebg-preview.png?raw=true' alt='DDS Logo'>"
        "<div id='header-text'>"
        "<h1>HALA BOT</h1>"
        "<p>Your Decoding Data Science (DDS) HR Leave Assistant</p>"
        "</div>"
        "</div>"
    )

    # Setup tabs for user layout
    with gr.Tabs():
        with gr.TabItem("Ask HALA BOT"):
            # Chat component with copy button enabled
            chatbot = gr.Chatbot(
                value=WELCOME_HISTORY,
                type="messages",
                height=400,
                label="HALA BOT Chat",
                show_copy_button=True
            )

            with gr.Row():
                box = gr.Textbox(
                    placeholder="Ask about carry-over days, maternity notice, sick leave...",
                    show_label=False,
                    scale=4
                )
                send = gr.Button("Ask", variant="primary", scale=1)
                clear = gr.Button("Clear Chat", variant="stop", scale=1)

            # Clickable examples
            gr.Examples(examples=EXAMPLES, inputs=box, label="Frequently Asked Leave Questions")

            # Collapsible sources panel
            with gr.Accordion("🔍 Where did this answer come from? (Sources)", open=False):
                sources_box = gr.Markdown("Ask a question to see the policy chunks used.")

        with gr.TabItem("About"):
            gr.Markdown("""
            ### About HALA BOT & DDS Leave Policy

            **What HALA BOT Can Answer:**
            * Annual leave entitlement, carry-over rules, and notice periods.
            * Parental leave, maternity, paternity, and bereavement policies.
            * Public holiday policies and medical certificate guidelines.

            **What HALA BOT Cannot Answer:**
            * Personal leave balances (Coming tomorrow in Day 3!).
            * Discretionary exceptions outside of the general written policy.

            ---

            **Policy Specifications:**
            * **Document ID:** DDS-HR-POL-001
            * **Version:** 2.0
            * **Effective Date:** 1 July 2026

            ---

            **Need more help?**
            If your question is sensitive or requires manual approval, please reach out directly to the **People and Culture** team via email or Slack.
            """)

    # Trustworthy disclaimer in footer
    gr.Markdown(
        "*Disclaimer: HALA BOT provides guidance based strictly on the DDS Leave Policy v2.0. "
        "For official or final determinations, please consult directly with the People & Culture department.*",
        elem_id="footer"
    )

    # Set up events
    send.click(respond, [box, chatbot], [box, chatbot, sources_box])
    box.submit(respond, [box, chatbot], [box, chatbot, sources_box])
    clear.click(clear_session, outputs=[chatbot, sources_box])

app.launch(share=True)


Closing server running on port: 7861


/tmp/ipykernel_1310/2331665701.py:113: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=deco_theme, css=CSS, title="HALA BOT | DDS HR Assistant") as app:
/tmp/ipykernel_1310/2331665701.py:113: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=deco_theme, css=CSS, title="HALA BOT | DDS HR Assistant") as app:
/tmp/ipykernel_1310/2331665701.py:129: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_1310/2331665701.py:129: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in y

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/11/28 10:54:22 [W] [service.go:132] login to server failed: session shutdown


<IPython.core.display.Javascript object>

---
## Part E | Write a professional prompt (12 min)

A prompt is the **instruction manual** you give the AI together with the chunks. A weak prompt gives weak answers, even with perfect search.

### Six things that make a prompt professional

| # | Ingredient | Example for DecoBot |
|---|---|---|
| 1 | **Role** | You are DecoBot, the leave assistant for DDS. |
| 2 | **Source rule** | Answer only from the policy excerpts. Never use outside knowledge. |
| 3 | **Not-found line** | If the answer is not in the excerpts, say exactly: "I could not find that..." |
| 4 | **Exact numbers** | Quote numbers and dates exactly as written. |
| 5 | **Format** | One sentence answer first, then at most 3 short bullets, then the page number. |
| 6 | **Tone** | Warm, clear, professional. Plain words. |

### Step 13: Basic prompt versus professional prompt

*We run the **same three questions** through two prompts. The search is identical. Only the instructions change.
Look at the third question: the policy says nothing about the dress code.*

In [15]:
from llama_index.core import PromptTemplate

BASIC_PROMPT = PromptTemplate(
    "Use the excerpts to answer the question.\n"
    "---------------------\n{context_str}\n---------------------\n"
    "Question: {query_str}\nAnswer: "
)

PRO_PROMPT = PromptTemplate(
    "You are DecoBot, the friendly leave assistant for Decoding Data Science (DDS).\n"
    "You help employees understand the DDS Leave Policy, version 2.\n\n"
    "RULES\n"
    "1. Answer ONLY from the policy excerpts below. Never use outside knowledge.\n"
    "2. Quote numbers and dates exactly as written in the excerpts.\n"
    "3. If the answer is not in the excerpts, reply exactly: "
    "'I could not find that in the DDS leave policy. Please contact People & Culture.'\n"
    "4. Never guess and never invent a rule.\n\n"
    "STYLE\n"
    "- Warm, clear and professional. Plain words, no jargon.\n"
    "- Start with the direct answer in one sentence.\n"
    "- Then give at most 3 short bullet points with the key details.\n"
    "- Finish with a line like: Source: policy page 3.\n\n"
    "POLICY EXCERPTS\n---------------------\n{context_str}\n---------------------\n"
    "EMPLOYEE QUESTION: {query_str}\n"
    "YOUR ANSWER:"
)

def compare(question):
    print("=" * 70)
    print("QUESTION:", question)
    for label, prompt in [("BASIC", BASIC_PROMPT), ("PROFESSIONAL", PRO_PROMPT)]:
        answer = str(make_engine(prompt).query(question)).strip()
        print(f"\n--- {label} ---\n{answer}")

compare("How much notice do I need to book 12 working days of annual leave?")
compare("My grandmother passed away. How many days of leave do I get?")
compare("What is the dress code at DDS?")

QUESTION: How much notice do I need to book 12 working days of annual leave?

--- BASIC ---
You need to give **30 calendar days’ notice** to book **12 working days** of annual leave, since requests for **10 or more working days** require **30 calendar days’ notice**.

--- PROFESSIONAL ---
You need 30 calendar days' notice.

- For 10 or more working days requested, the notice period is 30 calendar days' notice.
- Managers respond to leave requests within 3 business days.
- Please do not book non-refundable travel before approval is confirmed in the leave system.

Source: policy page 1.
QUESTION: My grandmother passed away. How many days of leave do I get?

--- BASIC ---
You get **3 working days of bereavement leave at full pay** for the death of a **grandparent**.

--- PROFESSIONAL ---
You get **3 working days at full pay**.

- This applies to the death of a **grandparent**.
- You should **notify your manager as soon as possible**.
- **Documentation may be requested**.

Source: policy p

**Checkpoint.** Compare the two answers for each question.

- Which one is easier to read in five seconds?
- For the dress code: which one admitted that the policy is silent?
- Which one tells the employee where to check?

### Step 14: Improve the prompt yourself

*Below is the professional prompt as plain text. **Change it**: add your own rule, change the tone, add a language rule, ask for a different format.
Then run the cell to test it on two questions. Ideas are listed under the cell.*

In [16]:
MY_PROMPT_TEXT = """You are DecoBot, the friendly leave assistant for Decoding Data Science (DDS).
You help employees understand the DDS Leave Policy, version 2.

RULES
1. Answer ONLY from the policy excerpts below. Never use outside knowledge.
2. Quote numbers and dates exactly as written in the excerpts.
3. If the answer is not in the excerpts, reply exactly: I could not find that in the DDS leave policy. Please contact People & Culture.
4. Never guess and never invent a rule.

STYLE
- Warm, clear and professional. Plain words, no jargon.
- Start with the direct answer in one sentence.
- Then give at most 3 short bullet points with the key details.
- Finish with a line like: Source: policy page 3.
-Speak like a calm, helpful HR colleague." versus "Be formal and brief.

POLICY EXCERPTS
---------------------
{context_str}
---------------------
EMPLOYEE QUESTION: {query_str}
YOUR ANSWER:"""

MY_PROMPT = PromptTemplate(MY_PROMPT_TEXT)    # keep {context_str} and {query_str} in the text

for question in ["I want to take 5 days of annual leave next month. What should I do?",
                 "Can I get my unused leave paid out in cash?"]:
    print("QUESTION:", question)
    print(str(make_engine(MY_PROMPT).query(question)).strip(), "\n")

QUESTION: I want to take 5 days of annual leave next month. What should I do?
You should submit your annual leave request in the leave system with your dates and a short coverage plan, then wait for approval.

- For 5 working days of annual leave, the notice period is 14 calendar days' notice.
- Managers respond to leave requests within 3 business days.
- Employees should not book non-refundable travel before approval is confirmed in the leave system.

Source: policy page 1. 

QUESTION: Can I get my unused leave paid out in cash?
Unused annual leave is not paid out during employment.

- On termination of employment, accrued but unused annual leave for the current year is paid out at the employee's basic salary rate, in line with UAE Labour Law.
- Carry-over is automatic, and a maximum of 10 unused annual-leave days may be carried into the next leave year.
- Any carried-over days not used by 31 March expire without compensation.

Source: policy page 2. 



**Ideas to try (change one thing at a time and compare):**

- **Tone:** "Speak like a calm, helpful HR colleague." versus "Be formal and brief."
- **Format:** "Answer in a numbered list" or "Answer in 2 sentences maximum."
- **Language:** "If the employee writes in Arabic, answer in Arabic."
- **Next step:** "End with one sentence telling the employee what to do next."
- **Escalation:** "For medical or personal situations, add: please speak to People & Culture."

**Or let an AI improve your prompt.** Paste your prompt into an AI assistant with this:
```
Here is the prompt for an HR leave assistant that answers only from a policy document. Review it as a prompt engineer.
List the 5 weakest points, then rewrite it so answers are shorter, safer and more professional.
Keep {context_str} and {query_str} exactly as they are.
```

### Step 15: Put your prompt into the app

*The chat app calls `ask()`, and `ask()` uses the variable `engine`. When we replace `engine`, the **running app switches to your new prompt**. No need to rebuild the app.
Then go back to the app above and ask the dress code question again.*

In [17]:
engine = make_engine(MY_PROMPT)          # the app now uses your professional prompt

answer, sources = ask("What is the dress code at DDS?")
print(answer)

# If you closed the app, run the Step 12 cell again to open it.

I could not find that in the DDS leave policy. Please contact People & Culture.


**Checkpoint.** Ask three questions in the app: one the policy answers, one it does not, and one with a personal situation.
Does the app now sound like a professional HR assistant?

---
## Wrap-up (2 min)

### What we built today
```
PDF -> chunks of 300 tokens -> numbers (embeddings) -> saved in a folder -> search with scores -> AI answer -> Gradio chat app
```

### Three things to remember
1. **Chunks decide what search can find.** Look at the retrieved chunks first when an answer is wrong.
2. **The saved folder is just files.** Build once, load many times.
3. **A professional prompt** has a role, a source rule, a not-found line, a format and a tone.

### Self-check
1. What is a token? What does `CHUNK_SIZE = 300` mean?
2. What is stored in the `decobot_index` folder?
3. Which line of the prompt stops the bot from inventing an answer?

### Parked for later (not today)
Cutting the policy by section, measuring search quality with test questions, common RAG failure modes, and hosted vector databases such as Pinecone
(optional take-home notebook). We will use them when you are ready.

### Tomorrow (Day 3)
SQL on the employee leave database, tools, and an agent that decides when to read the policy and when to check your balance.